# Phase 2: Preprocessing — Videos → .npy Tensors

**Kaggle settings required:**
- Internet ON (to install `av`)
- Accelerator: None (CPU is enough; GPU saves no time here)
- Add the output of Notebook 01 as an **input dataset**

**Key correctness decisions:**
- Saves tensors **WITHOUT normalization** — normalization is applied at train time and baked into the model during export
- Shape per file: `(1, 3, 16, 112, 112)` — float32, values in `[0, 1]`
- Uses `decord` (fast) rather than `VideoClips` (slow) for frame extraction
- Writes `manifest.jsonl` that the training notebook reads

**After running:** Save `/kaggle/working/preprocessed_tensors` as a Kaggle Dataset.

In [ ]:
import os, json, time, numpy as np, torch
import torchvision.transforms as T
from decord import VideoReader, cpu
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
# Install decord — much faster than VideoClips for frame extraction on Kaggle
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'decord'], check=True)
print('✅ decord installed')

In [ ]:
# Replace YOUR-DATASET-NAME with the slug shown when you add
# the QEVD-Dataset-Creation output as a Kaggle input dataset.
DATASET_INPUT   = '/kaggle/input/qevd-creation-output'
DATA_ROOT       = os.path.join(DATASET_INPUT, 'qevd_final')

OUT_ROOT        = '/kaggle/working/preprocessed_tensors'
CLIP_LEN        = 16      # frames per clip
TARGET_FPS      = 4       # resample video to this FPS before sampling
MAX_STORAGE_GB  = 18.0    # stop before hitting Kaggle 20 GB limit

os.makedirs(OUT_ROOT, exist_ok=True)

# Guard: crash early if paths are wrong
assert os.path.isdir(DATA_ROOT), (
    f'DATA_ROOT not found: {DATA_ROOT}\n'
    f'Update DATASET_INPUT to match your Kaggle input dataset slug.'
)
print('✅ Paths validated')

In [ ]:
# =========================================================
# DISCOVER CLASSES & PERFORM SPLIT
# =========================================================
from sklearn.model_selection import train_test_split

# 1. Discover classes from subfolders
classes = sorted([
    d for d in os.listdir(DATA_ROOT) 
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])
class_map = {name: i for i, name in enumerate(classes)}
class_labels = {str(i): name for i, name in enumerate(classes)}

print(f'✅ {len(classes)} classes discovered')

# 2. Save class mapping for reference (used by Training and Export notebooks)
with open(os.path.join(OUT_ROOT, 'class_map.json'), 'w') as f:
    json.dump(class_map, f, indent=2)
with open(os.path.join(OUT_ROOT, 'class_labels.json'), 'w') as f:
    json.dump(class_labels, f, indent=2)

# 3. Gather all video files with their labels for split
vids_list = []
labels_list = []
for cls in classes:
    cls_dir = os.path.join(DATA_ROOT, cls)
    vids = sorted([f for f in os.listdir(cls_dir) if f.endswith('.mp4')])
    
    if not vids:
        continue
        
    if len(vids) == 1:
        # 🔥 RARE CLASS AUGMENTATION: Duplicate the single video path
        # This allows stratify=labels_list to work (needs at least 2 samples)
        rel_path = os.path.join(cls, vids[0])
        vids_list.extend([rel_path, rel_path])
        labels_list.extend([cls, cls])
        print(f'🔔 Augmented rare class "{cls}" (duplicated 1 sample for split)')
    else:
        for f in vids:
            vids_list.append(os.path.join(cls, f))
            labels_list.append(cls)

# 4. Perform stratified 85/15 split
train_files, val_files = train_test_split(
    vids_list, test_size=0.15, stratify=labels_list, random_state=42
)

split_files = {'train': train_files, 'val': val_files}
print(f'✅ Stratified split: {len(train_files)} train, {len(val_files)} val')

In [ ]:
# =========================================================
# SPATIAL TRANSFORMS — no normalization, just resize + crop
# Normalization is applied at TRAIN TIME and baked into the
# model graph during QNN export.
# =========================================================
spatial = T.Compose([
    T.ConvertImageDtype(torch.float32),   # uint8 [0,255] -> float32 [0,1]
    T.Resize((128, 171), antialias=False), # match competition spec (H, W)
    T.CenterCrop((112, 112)),
])


def extract_clip(video_path: str) -> np.ndarray | None:
    """
    Extract a single 16-frame clip sampled at 4 fps using decord.
    Returns an ndarray of shape (1, 3, 16, 112, 112) in float32 [0,1]
    or None if the video is too short / corrupt.
    """
    try:
        vr = VideoReader(video_path, ctx=cpu(0))
        native_fps = vr.get_avg_fps()
        total_frames = len(vr)

        # Compute frame indices at TARGET_FPS
        step = max(1, round(native_fps / TARGET_FPS))
        indices = list(range(0, total_frames, step))

        if len(indices) < CLIP_LEN:
            return None  # video too short

        # Take the first CLIP_LEN frames (matches UniformClipSampler at t=0)
        indices = indices[:CLIP_LEN]

        frames = vr.get_batch(indices).asnumpy()  # (T, H, W, C) uint8
        t = torch.from_numpy(frames).permute(0, 3, 1, 2)
        t = spatial(t)           # (T, C=3, 112, 112) float32 [0,1]
        t = t.permute(1, 0, 2, 3).unsqueeze(0)  # (1, 3, 16, 112, 112)
        return t.numpy()

    except Exception:
        return None


def extract_augmented_clips(video_path: str, num_clips: int) -> list[np.ndarray]:
    """
    Extract multiple augmented clips from a single video so that
    under-represented classes get more training data.

    Augmentation strategies:
      1. Different temporal starting points (begin, middle, end)
      2. Horizontal flip of each temporal clip

    Returns a list of ndarrays, each (1, 3, 16, 112, 112) float32.
    """
    try:
        vr = VideoReader(video_path, ctx=cpu(0))
        native_fps = vr.get_avg_fps()
        total_frames = len(vr)
        step = max(1, round(native_fps / TARGET_FPS))
        all_indices = list(range(0, total_frames, step))

        if len(all_indices) < CLIP_LEN:
            return []

        clips = []
        max_start = len(all_indices) - CLIP_LEN

        # Generate evenly-spaced temporal offsets
        n_temporal = min((num_clips + 1) // 2, max_start + 1)  # half from temporal, half from flips
        if n_temporal <= 1:
            starts = [0]
        else:
            starts = np.linspace(0, max_start, n_temporal, dtype=int).tolist()

        for s in starts:
            sel = all_indices[s : s + CLIP_LEN]
            frames = vr.get_batch(sel).asnumpy()
            t = torch.from_numpy(frames).permute(0, 3, 1, 2)
            t = spatial(t)
            t = t.permute(1, 0, 2, 3).unsqueeze(0)  # (1, 3, 16, 112, 112)
            clips.append(t.numpy())

            if len(clips) >= num_clips:
                break

            # Horizontal flip of the same clip
            flipped = np.flip(t.numpy(), axis=-1).copy()  # flip W axis
            clips.append(flipped)

            if len(clips) >= num_clips:
                break

        return clips[:num_clips]

    except Exception:
        return []


print('Processing functions ready (single-clip + augmented multi-clip)')

In [ ]:
# =========================================================
# PROCESS ALL VIDEOS (BALANCED + AUGMENTED FOR SMALL CLASSES)
# =========================================================
manifest_path = os.path.join(OUT_ROOT, 'manifest.jsonl')
max_bytes     = int(MAX_STORAGE_GB * 1024 ** 3)
used_bytes    = 0
processed = skipped = augmented_total = 0
t0 = time.time()

MAX_VIDS_PER_CLASS = {'train': 75, 'val': 15}
MIN_VIDS_TARGET    = 30   # augment if class has fewer videos than this

with open(manifest_path, 'w') as manifest:
    for split in ['train', 'val']:
        videos_in_split = split_files[split]
        print(f'\n[{split}] Processing {len(videos_in_split)} videos')

        # Group files by class for easy capping/augmentation logic
        files_by_class = {}
        for f in videos_in_split:
            cls = os.path.dirname(f)
            if cls not in files_by_class: files_by_class[cls] = []
            files_by_class[cls].append(f)

        for cls_name in tqdm(classes, desc=split):
            videos = files_by_class.get(cls_name, [])
            if not videos: continue

            cls_out = os.path.join(OUT_ROOT, split, cls_name)
            os.makedirs(cls_out, exist_ok=True)

            n_available = len(videos)
            cap = MAX_VIDS_PER_CLASS[split]
            need_augment = (split == 'train' and n_available < MIN_VIDS_TARGET)

            if need_augment:
                clips_per_vid = max(1, MIN_VIDS_TARGET // n_available)
            else:
                clips_per_vid = 1
                videos = videos[:cap]

            for rel_vpath in videos:
                if used_bytes >= max_bytes: break

                full_vpath = os.path.join(DATA_ROOT, rel_vpath)
                fname = os.path.basename(rel_vpath)

                if clips_per_vid == 1:
                    arr = extract_clip(full_vpath)
                    if arr is None:
                        skipped += 1
                        continue
                    arrays = [arr]
                else:
                    arrays = extract_augmented_clips(full_vpath, clips_per_vid)
                    if not arrays:
                        skipped += 1
                        continue
                    augmented_total += len(arrays) - 1

                for ci, arr in enumerate(arrays):
                    if used_bytes >= max_bytes: break

                    stem = fname.replace('.mp4', '')
                    suffix = f'_aug{ci}' if ci > 0 else ''
                    out_path = os.path.join(cls_out, f'{stem}{suffix}.npy')
                    np.save(out_path, arr)

                    used_bytes += os.path.getsize(out_path)
                    manifest.write(json.dumps({
                        'label':       cls_name,
                        'split':       split,
                        'tensor_path': out_path,
                        'shape':       list(arr.shape),
                        'dtype':       'float32',
                    }) + '\n')
                    processed += 1

print(f'\n🚀 Preprocessing complete!')

In [ ]:
# =========================================================
# VERIFY OUTPUT
# =========================================================
import glob

npy_files = glob.glob(os.path.join(OUT_ROOT, '**', '*.npy'), recursive=True)
print(f'Total .npy files: {len(npy_files)}')

if npy_files:
    s = np.load(npy_files[0])
    print(f'Shape : {s.shape}')   # expect (1, 3, 16, 112, 112)
    print(f'Dtype : {s.dtype}')   # expect float32
    print(f'Min   : {s.min():.4f}')  # expect ≥ 0
    print(f'Max   : {s.max():.4f}')  # expect ≤ 1
    if s.min() >= -0.01 and s.max() <= 1.01:
        print('✅ Values in [0,1] — UN-normalized (correct)')
    else:
        print('⚠️  Values outside [0,1] — check transforms!')

for split in ['train', 'val']:
    cnt = len([f for f in npy_files if f'/{split}/' in f or f'\\{split}\\' in f])
    print(f'{split}: {cnt} tensors')

print('\n🚀 Save /kaggle/working as a Kaggle Dataset, then run Training notebook!')